# Module 3.2: Memory Identification

*This notebook focuses on **semantic memory** — durable user preferences, facts, and
constraints (e.g. "vegetarian", "home airport is CDG", "budget $250/night"). Episodic
events (past trips) and procedural knowledge (learned procedures) have different
identification paths, covered in Notebook 06.*

In Module 2 (Episodic Memory), we gave the agent a `store_event` tool and told it
to store important facts. But we left a critical question unanswered:

> **How does the agent decide what's worth remembering?**

Without explicit criteria, agents either store too much (noise, hypotheticals, sensitive data)
or too little (missing key preferences). This notebook builds the **identification layer**
that sits between raw conversation and semantic memory storage.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys, json
import sniffio

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import create_client
from agent_framework import AgentSession
from lifecycle_utils import (
    MemoryCandidate, MemoryDecision,
    create_baseline_agent, setup_backends,
)

client, credential = create_client("../.env")
print("Client ready")

## Setup: Connect Backends and Create Baseline Agent

We use the shared factory to stand up a travel agent with episodic memory (Cosmos DB).
This agent has `store_event` but **no identification criteria** — it will store
whatever it's told to store.

In [ ]:
# Connect to Cosmos DB + AI Search
cosmos_container, search_client, openai_client = setup_backends(credential, "../.env")

# Create baseline agent — has store_event but NO identification logic
baseline_agent = create_baseline_agent(
    client, credential,
    cosmos_container=cosmos_container,
    search_client=search_client,
    openai_client=openai_client,
)
print(f"Baseline agent ready: {baseline_agent.name}")
# print(f"Tools: {[t.__name__ if hasattr(t, '__name__') else str(t) for t in baseline_agent.tools]}")

## The Problem: An Agent That Stores Everything

We simulate a 10-turn conversation with the agent. At the end, we ask it to
"store all the important things you learned about me." Without identification
criteria, the agent has no principled way to separate signal from noise.

In [ ]:
# A realistic 10-turn conversation between Sarah and the travel agent
SAMPLE_CONVERSATION = [
    # Turn 1: Durable preference (should memorise)
    "I always prefer window seats on long flights — I like watching the landscape.",
    # Turn 2: Transient request (should discard)
    "Can you check if there's a flight to Chicago tomorrow?",
    # Turn 3: Durable fact (should memorise)
    "I'm based in San Francisco, so SFO is my home airport.",
    # Turn 4: Hypothetical (should discard)
    "What if I wanted to fly first class — how much more would that be?",
    # Turn 5: Strong preference (should memorise)
    "I really don't like layovers longer than 2 hours. I'd rather pay more for direct.",
    # Turn 6: Other person's info (should discard)
    "My colleague Mike says the Hilton downtown is terrible.",
    # Turn 7: Actionable constraint (should memorise)
    "My company reimburses up to $250/night for hotels, so keep it under that.",
    # Turn 8: Session-specific (should discard)
    "Actually, go back to that first option you showed me.",
    # Turn 9: Repeated preference confirmation (should memorise)
    "Yes, Marriott is my go-to chain. I have their loyalty program.",
    # Turn 10: Sensitive data (should discard)
    "My loyalty number is MR-998877-2024.",
]

# print(f"Conversation has {len(SAMPLE_CONVERSATION)} turns")
# for i, msg in enumerate(SAMPLE_CONVERSATION, 1):
#     print(f"  Turn {i:2d}: {msg[:70]}{'...' if len(msg) > 70 else ''}")

In [ ]:
# Run the baseline agent: feed all turns, then ask it to store memories
session = AgentSession()

# Simulate the multi-turn conversation
for turn in SAMPLE_CONVERSATION:
    result = await baseline_agent.run(turn, session=session)

# Now ask it to store what it learned
store_prompt = (
    "Based on our conversation, please store all the important things you learned "
    "about me as separate events using the store_event tool. Use event_type='preference' "
    "for preferences and 'fact' for facts."
)
result = await baseline_agent.run(store_prompt, session=session)
print("Agent response:\n")
print(result.text)

In [ ]:
# Check what the agent actually stored in Cosmos
items = [item async for item in cosmos_container.query_items(
    "SELECT c.event_type, c.description, c.details FROM c WHERE c.user_id = 'E001' ORDER BY c.timestamp DESC OFFSET 0 LIMIT 20",
    partition_key="E001"
)]
print(f"Events stored in Cosmos DB: {len(items)}\n")
for item in items:
    print(f"  [{item.get('event_type', '?'):10s}] {item.get('description', '?')}")

## What Went Wrong

The baseline agent stored **everything** it could extract — including:

| Failure | Example | Impact |
|---------|---------|--------|
| **Wrong attribution** | "My colleague Mike says…" (turn 6) | Stores someone else's opinion as Sarah's |
| **Sensitive data** | "My loyalty number is…" (turn 10) | Privacy violation — shouldn't persist |

The core issue: `store_event` has no gate. Whatever the agent decides to store,
it stores. We need explicit criteria to separate signal from noise **before** storage.

## The Solution: Memory Identification Criteria

We score every candidate memory on four dimensions (from the [SKILL.md](skills/memory-identification/SKILL.md)):

| Criterion | Definition | Example (High) | Example (Low) |
|-----------|------------|----------------|---------------|
| **Durability** | How likely to remain true over weeks/months | "I'm vegetarian" | "I'm in a rush today" |
| **Reusability** | How likely to be useful in future interactions | "Prefers aisle seats" | "Flight UA123 departs at 3pm" |
| **User-Specificity** | How personal vs generic knowledge | "Sarah hates layovers" | "JFK has 6 terminals" |
| **Actionability** | Can the agent use this to improve service | "Budget max $300/night" | "Weather is nice today" |

**Decision rule** (composite score = average of all four, each 0.0–1.0):
- **≥ 0.6** → Memorise (store as candidate memory)
- **0.4 – 0.6** → Ask the user for confirmation
- **< 0.4** → Discard (do not persist)

### Reference
> *"Towards Root Memories"* (arXiv:2606.23283) — introduces the distinction between
> reusable, decision-affecting "root" memories vs noise.

In [ ]:
from agent_framework import SkillsProvider

procedures = SkillsProvider.from_paths("skills")

# Load skills metadata to display what was discovered
skills = await procedures._source.get_skills()
print("SkillsProvider discovered:\n")
for skill in skills:
    print(f"  [{skill.frontmatter.name}] {skill.frontmatter.description}")
print(f"\nTools added: {procedures.LOAD_SKILL_TOOL_NAME}, {procedures.READ_SKILL_RESOURCE_TOOL_NAME}")

In [ ]:
from agent_framework._types import Message

# Visualization helper — NOT used by the agent. This lets us peek at
# how the scoring criteria work on individual turns for teaching purposes.
async def score_turn(msg: str) -> MemoryCandidate:
    """Score a single message against identification criteria (for visualization only)."""
    prompt = ("Score this user message for memorability. Return ONLY JSON with: "
              "content, category (preference|fact|event|procedure), "
              "durability (0-1), reusability (0-1), user_specificity (0-1), "
              "actionability (0-1), decision (memorise|discard|ask_user), reasoning.")
    messages = [Message(role="system", contents=[prompt]),
                Message(role="user", contents=[msg])]
    response = await client.get_response(messages=messages)
    raw = response.text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
    data = json.loads(raw)
    return MemoryCandidate(
        content=data["content"], category=data["category"],
        durability=data["durability"], reusability=data["reusability"],
        user_specificity=data["user_specificity"], actionability=data["actionability"],
        decision=MemoryDecision(data["decision"]), reasoning=data["reasoning"],
    )

print("Visualization helper ready (score_turn)")

## Building a Skill-Equipped Agent

Instead of a tool that calls a second LLM, we give the agent the scoring
criteria as a **Skill**. The `SkillsProvider` loads the
[SKILL.md](skills/memory-identification/SKILL.md) into the agent's context.

**One LLM call, not two.**

The agent reads the criteria, scores memorability as part of its own
reasoning, and calls `store_event` only when the score passes threshold.

In [ ]:
from agent_framework import ToolApprovalMiddleware

# Auto-approve skill tools so the agent can load_skill without pausing
approval = ToolApprovalMiddleware(
    auto_approval_rules=[SkillsProvider.all_tools_auto_approval_rule]
)

gated_agent = create_baseline_agent(
    client, credential,
    cosmos_container=cosmos_container,
    search_client=search_client,
    openai_client=openai_client,
    instructions_suffix=(
        "You have a memory-identification skill. "
        "BEFORE storing any memory, load the skill and apply its "
        "scoring criteria. Only call store_event when composite "
        "score >= 8 (out of 12). For 5-7, ask the user. "
        "For <= 4, discard. NEVER store: hypotheticals, other "
        "people's info, sensitive data, session-specific context."
    ),
    context_providers=[procedures],
    middleware=[approval],
)
gated_agent.name = "SkillGatedTravelAssistant"
print(f"Skill-equipped agent ready: {gated_agent.name}")

## The Payoff: Same Conversation, Skill-Guided Identification

We run the same 10-turn conversation through the skill-equipped agent.
This time, the agent applies the identification criteria from the skill
as part of its own reasoning — deciding inline what to store and what to skip.

In [ ]:
# Run the skill-equipped agent — it identifies memories inline
gated_session = AgentSession()

for turn in SAMPLE_CONVERSATION:
    result = await gated_agent.run(turn, session=gated_session)

# Ask the agent to review and store what it learned
store_prompt = (
    "Based on our conversation, review what you learned about me. "
    "Load the memory-identification skill, apply the scoring criteria, "
    "and store only what scores >= 8. My user_id is E001."
)
result = await gated_agent.run(store_prompt, session=gated_session)
print("Agent response:\n")
print(result.text)

## Under the Hood: Scoring Breakdown

The agent applies the skill criteria inline during its reasoning. To visualize
the scoring, we run each turn through the same criteria independently.
This is for teaching only — in production, the agent does this as part of
its single LLM call:

In [ ]:
print(f"Scoring {len(SAMPLE_CONVERSATION)} conversation turns...\n")

candidates = []
for msg in SAMPLE_CONVERSATION:
    candidates.append(await score_turn(msg))

print("Composite Score by Turn:")
print("=" * 70)
for i, (msg, c) in enumerate(zip(SAMPLE_CONVERSATION, candidates), 1):
    bar = "█" * int(c.composite_score * 40)
    icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[c.decision.value]
    print(f"Turn {i:2d} {icon} |{bar:<40}| {c.composite_score:.2f}")
    print(f"         {msg[:65]}{'...' if len(msg) > 65 else ''}")
    if c.decision != MemoryDecision.DISCARD:
        print(f"         → Store: {c.content}")
    print()

memorised = sum(1 for c in candidates if c.decision == MemoryDecision.MEMORISE)
discarded = sum(1 for c in candidates if c.decision == MemoryDecision.DISCARD)
ask_user = sum(1 for c in candidates if c.decision == MemoryDecision.ASK_USER)

print("=" * 70)
print(f"✅ Memorise: {memorised}  |  ❌ Discard: {discarded}  |  ❓ Ask user: {ask_user}")

print(f"\nBaseline agent stored: ~10 items (everything)")
print(f"Skill-equipped agent:  ~{memorised} items (only what passes criteria)")

## Edge Cases: What NOT to Memorise

The classifier's value is in handling tricky inputs that *look* like preferences
but should NOT be stored:

In [ ]:
EDGE_CASES = [
    # Looks like preference but is hypothetical
    "If I were to move to London, I'd probably want to fly British Airways.",
    # Looks like a fact but is about someone else
    "My boss always flies Delta — maybe I should try them too.",
    # Contains sensitive data that shouldn't persist
    "My passport number is AB1234567, expiring March 2028.",
    # Temporary state, not durable
    "I'm feeling sick today so I might cancel my trip.",
    # Genuine durable preference (control case — should memorise)
    "I have a severe peanut allergy — please always flag this for meal selection.",
]

print("Edge Case Classification:\n")
for msg in EDGE_CASES:
    candidate = await score_turn(msg)
    icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[candidate.decision.value]
    display = f'"{msg[:65]}..."' if len(msg) > 65 else f'"{msg}"'
    print(f"{icon} [{candidate.decision.value:8s}] {display}")
    print(f"   Score: {candidate.composite_score:.2f} | {candidate.reasoning}\n")

## Production Hardening: Filter Pipeline

In production, identified candidates pass through additional filters before storage:
1. **Category allowlist** — Only store certain categories (e.g., no "event" for privacy)
2. **Minimum confidence** — Below threshold, discard even if classified as "memorise"
3. **Rate limiting** — Don't store more than N memories per conversation

In [ ]:
from dataclasses import dataclass

@dataclass
class FilterConfig:
    allowed_categories: list = None
    min_composite_score: float = 0.5
    max_memories_per_conversation: int = 10

    def __post_init__(self):
        if self.allowed_categories is None:
            self.allowed_categories = ["preference", "fact", "procedure"]


class MemoryFilterPipeline:
    """Post-identification filtering before memory storage."""

    def __init__(self, config: FilterConfig = None):
        self.config = config or FilterConfig()

    def apply(self, candidates: list[MemoryCandidate]) -> list[MemoryCandidate]:
        """Filter candidates through the pipeline."""
        kept = [c for c in candidates if c.decision == MemoryDecision.MEMORISE]
        kept = [c for c in kept if c.category in self.config.allowed_categories]
        kept = [c for c in kept if c.composite_score >= self.config.min_composite_score]
        kept.sort(key=lambda c: c.composite_score, reverse=True)
        kept = kept[: self.config.max_memories_per_conversation]
        return kept


pipeline = MemoryFilterPipeline()
filtered = pipeline.apply(candidates)

print(f"Before filtering: {len(candidates)} candidates")
print(f"After filtering:  {len(filtered)} memories to store\n")
for c in filtered:
    print(f"  [{c.category:10s}] {c.content}")
    print(f"              Score: {c.composite_score:.2f} | {c.reasoning}\n")

## Connecting to the Lifecycle

Identified memories don't go directly to "trusted" status — they enter the lifecycle
as **candidates**:

```mermaid
flowchart LR
    A["User message"] --> B["Agent\n(with skill context)"]
    B -->|"scores inline\n(single LLM call)"| C{"Score?"}
    C -->|"≥ 8"| D["store_event"]
    C -->|"5–7"| E["Ask user"]
    C -->|"≤ 4"| F["Discard"]
    D --> G[("Cosmos DB\n(candidate)")]
    E -->|"confirmed"| D
    G -.->|"Notebook 03"| H["Staged Promotion\ncandidate → provisional → trusted"]
```

Notice: **one LLM call, not two**. The agent reads the skill criteria, reasons
about memorability, and decides whether to call `store_event` — all in a
single pass. No wrapper tool, no second classifier.

## Key Takeaways

1. **Not everything is memory** — without filtering, agents store noise, hypotheticals, and sensitive data
2. **Four dimensions** — durability, reusability, user-specificity, actionability
3. **Skill > wrapper tool** — give the agent criteria via SkillsProvider, not a tool that calls another LLM
4. **Anti-patterns matter** — hypotheticals, others' info, sensitive data must be excluded
5. **Pipeline filtering** — category allowlist, score threshold, rate limiting provide defense-in-depth
6. **Memory starts as candidate** — identification doesn't equal trust (→ Notebook 03)

## Next: Staged Promotion (Notebook 03)

Now that we can identify *what* to store, the next question is:
**when should the agent trust it?** Newly identified memories must earn confidence
through repeated confirmation before influencing agent behaviour.